In [ ]:
# In[1]:

import pandas as pd

# Load files
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_error = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes (Unix seconds)
for df in (df_metric, df_trace, df_log, df_error):
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], unit="s", utc=True)

# Helper to build a compact one-row summary DataFrame
def build_summary_df(df, name_col=None, sample_messages=False):
    rows = len(df)
    min_ts = df["timestamp"].min() if rows > 0 else pd.NaT
    max_ts = df["timestamp"].max() if rows > 0 else pd.NaT
    cmdb_ids = sorted(df["cmdb_id"].dropna().unique().tolist())
    summary = pd.DataFrame([{
        "rows": rows,
        "min_timestamp_utc": min_ts,
        "max_timestamp_utc": max_ts,
        "distinct_cmdb_ids": cmdb_ids
    }])
    top_names = None
    samples = None
    if name_col and name_col in df.columns:
        top_names = (df.groupby(name_col).size()
                       .rename("count")
                       .reset_index()
                       .sort_values("count", ascending=False)
                       .head(20))
    if sample_messages and "message" in df.columns:
        samples = pd.DataFrame({
            "sample_messages": sorted(df["message"].dropna().unique())[:20]
        })
    return summary, top_names, samples

# Metric summary and top kpis
metric_summary, metric_top_kpis, _ = build_summary_df(df_metric, name_col="kpi_name")

# Trace summary and top trace names
trace_summary, trace_top, _ = build_summary_df(df_trace, name_col="trace_name")

# Log summary and top log names
log_summary, log_top, _ = build_summary_df(df_log, name_col="log_name")

# Error logs summary and up to 20 sample messages
error_summary, _, error_samples = build_summary_df(df_error, sample_messages=True)

# Keep outputs compact: show each summary (one-row DataFrame) and top lists (up to 20 rows)
metric_summary, metric_top_kpis, trace_summary, trace_top, log_summary, log_top, error_summary, error_samples

```
Out[1]:
```
summary = (
    "metric.csv: 1899 rows, timestamps 2024-01-20 08:31:00+00:00 to 2024-01-20 08:55:00+00:00 UTC; "
    "distinct cmdb_ids include carts, carts-db, catalogue, catalogue-db, front-end, orders, ...; "
    "top KPI names (counts): cpu(375), mem(375), socket(375), diskio(199), latency-90(175), latency-50(175), workload(175), error(50).\n"
    "trace.csv: 0 rows (no timestamps, no trace names).\n"
    "log.csv: 456 rows, timestamps 2024-01-20 08:31:00+00:00 to 2024-01-20 08:55:00+00:00 UTC; "
    "distinct cmdb_ids include carts, carts-db, catalogue, front-end, orders, ...; "
    "top log names (counts): log.error_count(228), log.row_count(228).\n"
    "error_logs.csv: 0 rows (no sample messages)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(   rows         min_timestamp_utc         max_timestamp_utc                                  distinct_cmdb_ids
0  1899 2024-01-20 08:31:00+00:00 2024-01-20 08:55:00+00:00  [carts, carts-db, catalogue, catalogue-db, fro...,      kpi_name  count
0         cpu    375
5         mem    375
6      socket    375
1      diskio    199
4  latency-90    175
3  latency-50    175
7    workload    175
2       error     50,    rows min_timestamp_utc max_timestamp_utc distinct_cmdb_ids
0     0               NaT               NaT                [], Empty DataFrame
Columns: [trace_name, count]
Index: [],    rows         min_timestamp_utc         max_timestamp_utc                                  distinct_cmdb_ids
0   456 2024-01-20 08:31:00+00:00 2024-01-20 08:55:00+00:00  [carts, carts-db, catalogue, front-end, orders...,           log_name  count
0  log.error_count    228
1    log.row_count    228,    rows min_timestamp_utc max_timestamp_utc distinct_cmdb_ids
0     0               NaT               NaT                [], Empty DataFrame
Columns: [sample_messages]
Index: [])```
```

In [ ]:
# In[2]:

import pandas as pd

# Reuse existing df_metric if available, otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")

# Ensure timestamp parsed as UTC datetimes
df_metric["timestamp"] = pd.to_datetime(df_metric["timestamp"], unit="s", utc=True)

# Compute global P95 per (cmdb_id, kpi_name) using the entire series (before any window filtering)
p95_df = (
    df_metric
    .groupby(["cmdb_id", "kpi_name"], as_index=False)["value"]
    .quantile(0.95)
    .rename(columns={"value": "p95_threshold"})
)

# Merge p95 back onto full metric frame
df_metric_with_p95 = df_metric.merge(p95_df, on=["cmdb_id", "kpi_name"], how="left")

# Define incident window (UTC) and filter rows within it (inclusive)
window_start = pd.to_datetime("2024-01-20 08:28:56", utc=True)
window_end = pd.to_datetime("2024-01-20 08:58:56", utc=True)
df_window = df_metric_with_p95[
    (df_metric_with_p95["timestamp"] >= window_start) &
    (df_metric_with_p95["timestamp"] <= window_end)
].copy()

# Mark anomalies: value >= global P95 (use >= per instruction)
df_window["is_anomaly"] = df_window["value"] >= df_window["p95_threshold"]

# Create an anomaly timestamp column (timestamp when anomaly else NaT) to get earliest anomaly easily
df_window["anomaly_ts"] = df_window["timestamp"].where(df_window["is_anomaly"])

# Aggregate required metrics per (cmdb_id, kpi_name)
agg = (
    df_window
    .groupby(["cmdb_id", "kpi_name"], as_index=False)
    .agg(
        points_in_window=("value", "size"),
        anomalies_count=("is_anomaly", "sum"),
        earliest_anomaly_utc=("anomaly_ts", "min"),
        max_value_in_window=("value", "max"),
        p95_threshold=("p95_threshold", "first"),
    )
)

# Compact results: groups with anomalies_count > 0, sorted by anomalies_count desc (max 20 rows)
anomalies_df_top = (
    agg[agg["anomalies_count"] > 0]
    .sort_values(["anomalies_count", "max_value_in_window"], ascending=[False, False])
    .loc[:, ["cmdb_id", "kpi_name", "points_in_window", "anomalies_count", "earliest_anomaly_utc", "max_value_in_window", "p95_threshold"]]
    .head(20)
    .reset_index(drop=True)
)

# Up to 10 groups that had points in window but zero anomalies, sorted by max_value_in_window desc
zero_anoms_top = (
    agg[(agg["points_in_window"] > 0) & (agg["anomalies_count"] == 0)]
    .sort_values("max_value_in_window", ascending=False)
    .loc[:, ["cmdb_id", "kpi_name", "points_in_window", "anomalies_count", "max_value_in_window", "p95_threshold"]]
    .head(10)
    .reset_index(drop=True)
)

# Display the two compact tables (as variables for IPython)
anomalies_df_top, zero_anoms_top

```
Out[2]:
```
# Build a plain-English summary string from previous analysis results and display it.
try:
    n_anom_groups = anomalies_df_top.shape[0]
    top_rows = anomalies_df_top.head(5)
    zero_groups = zero_anoms_top.shape[0]
except NameError:
    # Fallback if variables aren't present (shouldn't happen in this session)
    import pandas as pd
    df_metric = pd.read_csv("metric.csv")
    df_metric["timestamp"] = pd.to_datetime(df_metric["timestamp"], unit="s", utc=True)
    summary = "Previous analysis results not found in the session."
    summary
else:
    lines = []
    lines.append(f"Found {n_anom_groups} (cmdb_id, kpi_name) groups with anomalies in the incident window (2024-01-20 08:28:56 to 2024-01-20 08:58:56 UTC).")
    lines.append("Top affected groups (up to 5), with anomalies_count, earliest anomaly UTC, max value in window, p95 threshold:")
    fmt = "{cmdb}/{kpi}: {anom} anomalies, earliest {ts}, max {mx}, p95 {p95}"
    for _, r in top_rows.iterrows():
        ts = pd.to_datetime(r["earliest_anomaly_utc"])
        ts_str = ts.strftime("%Y-%m-%d %H:%M:%S%z") if pd.notna(ts) else "N/A"
        mx = "{:.6g}".format(r["max_value_in_window"])
        p95 = "{:.6g}".format(r["p95_threshold"])
        lines.append(fmt.format(cmdb=r["cmdb_id"], kpi=r["kpi_name"], anom=int(r["anomalies_count"]), ts=ts_str, mx=mx, p95=p95))
    # High-level observations
    lines.append("")
    lines.append("Observations:")
    lines.append("- Many socket-related KPIs across services show continuous anomalies (most have anomalies_count = 25), indicating widespread socket-level issues affecting services such as user, rabbitmq, user-db, orders-db, carts-db, catalogue, etc.")
    lines.append("- rabbitmq-exporter mem has 25 anomalies with a very large max value (~4.73e+06), equal to its p95, meaning all points met or exceeded the global P95.")
    lines.append("- Several KPIs (diskio, error) show anomalies where both max and p95 are 0; these reflect thresholds of 0 and values equal to 0 (counts/metrics at zero) and should be interpreted cautiously.")
    if zero_groups > 0:
        lines.append(f"- Additionally, {zero_groups} groups had points in the window but zero anomalies (listed separately).")
    else:
        lines.append("- No groups had points in the window with zero anomalies (all groups with points showed anomalies).")
    summary = "\n".join(lines)
    summary

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id kpi_name  points_in_window  anomalies_count      earliest_anomaly_utc  max_value_in_window  p95_threshold
0   rabbitmq-exporter      mem                25               25 2024-01-20 08:31:00+00:00         4.730880e+06   4.730880e+06
1                user   socket                25               25 2024-01-20 08:31:00+00:00         2.300000e+01   2.300000e+01
2            rabbitmq   socket                25               25 2024-01-20 08:31:00+00:00         1.100000e+01   1.100000e+01
3             user-db   socket                25               25 2024-01-20 08:31:00+00:00         1.100000e+01   1.100000e+01
4           orders-db   socket                25               25 2024-01-20 08:31:00+00:00         9.000000e+00   9.000000e+00
5            carts-db   socket                25               25 2024-01-20 08:31:00+00:00         7.000000e+00   7.000000e+00
6           catalogue   socket                25               25 2024-01-20 08:31:00+00:00         7.000000e+00   7.000000e+00
7        catalogue-db   socket                25               25 2024-01-20 08:31:00+00:00         4.000000e+00   4.000000e+00
8        queue-master   socket                25               25 2024-01-20 08:31:00+00:00         3.000000e+00   3.000000e+00
9          session-db   socket                25               25 2024-01-20 08:31:00+00:00         3.000000e+00   3.000000e+00
10  rabbitmq-exporter   socket                25               25 2024-01-20 08:31:00+00:00         1.000000e+00   1.000000e+00
11       catalogue-db   diskio                25               25 2024-01-20 08:31:00+00:00         0.000000e+00   0.000000e+00
12          front-end    error                25               25 2024-01-20 08:31:00+00:00         0.000000e+00   0.000000e+00
13             orders    error                25               25 2024-01-20 08:31:00+00:00         0.000000e+00   0.000000e+00
14           rabbitmq   diskio                25               25 2024-01-20 08:31:00+00:00         0.000000e+00   0.000000e+00
15             orders   diskio                24               24 2024-01-20 08:32:00+00:00         0.000000e+00   0.000000e+00
16            payment   socket                25               22 2024-01-20 08:31:00+00:00         7.000000e+00   7.000000e+00
17              carts   socket                25                3 2024-01-20 08:31:00+00:00         1.200000e+01   1.200000e+01
18             orders      mem                25                2 2024-01-20 08:32:00+00:00         6.422760e+08   6.406069e+08
19       catalogue-db      mem                25                2 2024-01-20 08:34:00+00:00         4.305580e+08   4.305247e+08, Empty DataFrame
Columns: [cmdb_id, kpi_name, points_in_window, anomalies_count, max_value_in_window, p95_threshold]
Index: [])```
```

In [ ]:
# In[3]:

import pandas as pd

# Reuse existing df_log if available, otherwise load
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")

# Ensure timestamp parsed as UTC datetimes
df_log["timestamp"] = pd.to_datetime(df_log["timestamp"], unit="s", utc=True)

# Define incident window (UTC) inclusive
window_start = pd.to_datetime("2024-01-20 08:28:56", utc=True)
window_end = pd.to_datetime("2024-01-20 08:58:56", utc=True)

# Filter rows to the incident window (inclusive)
df_log_window = df_log[
    (df_log["timestamp"] >= window_start) &
    (df_log["timestamp"] <= window_end)
].copy()

# Compute earliest timestamp where value > 0 helper
def earliest_nonzero(series_ts, series_val):
    mask = series_val > 0
    if mask.any():
        return series_ts[mask].min()
    return pd.NaT

# Aggregate per (cmdb_id, log_name)
agg_logs = (
    df_log_window
    .groupby(["cmdb_id", "log_name"], as_index=False)
    .agg(
        points_in_window=("value", "size"),
        sum_value=("value", "sum"),
        max_value_in_window=("value", "max"),
        # earliest_nonzero_utc computed via apply on group
    )
)

# Compute earliest_nonzero_utc separately and merge to ensure correctness
earliest_nonzero_df = (
    df_log_window[df_log_window["value"] > 0]
    .groupby(["cmdb_id", "log_name"], as_index=False)
    .agg(earliest_nonzero_utc=("timestamp", "min"))
)

agg_logs = agg_logs.merge(earliest_nonzero_df, on=["cmdb_id", "log_name"], how="left")

# Sort by sum_value descending and keep top 20 rows for compactness
top_logs_table = agg_logs.sort_values("sum_value", ascending=False).head(20).reset_index(drop=True)

# Now produce a small separate table restricted to the five possible root-cause services
services = ["carts", "catalogue", "orders", "payment", "user"]

# Aggregate per service (cmdb_id) across all log_names in the window
svc_agg = (
    df_log_window
    .groupby("cmdb_id", as_index=False)
    .agg(
        points_in_window=("value", "size"),
        sum_value=("value", "sum"),
        max_value_in_window=("value", "max"),
    )
)

# earliest_nonzero per cmdb_id
svc_earliest = (
    df_log_window[df_log_window["value"] > 0]
    .groupby("cmdb_id", as_index=False)
    .agg(earliest_nonzero_utc=("timestamp", "min"))
)

svc_agg = svc_agg.merge(svc_earliest, on="cmdb_id", how="left")

# Restrict to the five services; include rows with zeros for services with no data
svc_agg_filtered = svc_agg[svc_agg["cmdb_id"].isin(services)].copy()

# Identify missing services and add zero rows for them
missing = [s for s in services if s not in svc_agg_filtered["cmdb_id"].tolist()]
if missing:
    missing_rows = pd.DataFrame({
        "cmdb_id": missing,
        "points_in_window": 0,
        "sum_value": 0.0,
        "max_value_in_window": 0.0,
        "earliest_nonzero_utc": pd.NaT
    })
    svc_agg_filtered = pd.concat([svc_agg_filtered, missing_rows], ignore_index=True)

# For clarity, mark these per-service rows as aggregated across all logs by setting log_name='ALL'
svc_agg_filtered = svc_agg_filtered.sort_values("sum_value", ascending=False).reset_index(drop=True)
svc_agg_filtered["log_name"] = "ALL"
svc_agg_table = svc_agg_filtered.loc[:, ["cmdb_id", "log_name", "points_in_window", "sum_value", "max_value_in_window", "earliest_nonzero_utc"]]

# Final compact outputs
top_logs_table, svc_agg_table

```
Out[3]:
```
summary = (
    "Within the incident window (2024-01-20 08:28:56 to 2024-01-20 08:58:56 UTC):\n"
    "- Top log activity by sum(log.row_count): front-end (sum=47003, points=25, max=2032, earliest nonzero=2024-01-20 08:31:00+00:00),\n"
    "  then user (14171, 25, max=644, earliest=2024-01-20 08:31:00+00:00), queue-master (13203, 25, max=639, earliest=2024-01-20 08:31:00+00:00),\n"
    "  catalogue (3894, 25, max=169, earliest=2024-01-20 08:31:00+00:00), orders (2934, 25, max=142, earliest=2024-01-20 08:31:00+00:00),\n"
    "  payment (2427, 25, max=111, earliest=2024-01-20 08:31:00+00:00), carts (1650, 24, max=150, earliest=2024-01-20 08:31:00+00:00).\n"
    "- log.error_count is present but sums to 0 for services in the window (no nonzero error_count observed).\n"
    "- Service-level aggregates for the five suspect services (combined across log names):\n"
    "  user: points=50, sum=14171, max=644, earliest_nonzero=2024-01-20 08:31:00+00:00;\n"
    "  catalogue: points=50, sum=3894, max=169, earliest_nonzero=2024-01-20 08:31:00+00:00;\n"
    "  orders: points=50, sum=2934, max=142, earliest_nonzero=2024-01-20 08:31:00+00:00;\n"
    "  payment: points=50, sum=2427, max=111, earliest_nonzero=2024-01-20 08:31:00+00:00;\n"
    "  carts: points=48, sum=1650, max=150, earliest_nonzero=2024-01-20 08:31:00+00:00.\n"
    "Conclusion: High log.row_count activity peaked around 2024-01-20 08:31:00 UTC across many services (front-end, user, queue-master, catalogue, orders, payment, carts). No nonzero log.error_count was recorded in the window. "
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(         cmdb_id         log_name  points_in_window  sum_value  max_value_in_window      earliest_nonzero_utc
0      front-end    log.row_count                25      47003                 2032 2024-01-20 08:31:00+00:00
1           user    log.row_count                25      14171                  644 2024-01-20 08:31:00+00:00
2   queue-master    log.row_count                25      13203                  639 2024-01-20 08:31:00+00:00
3      catalogue    log.row_count                25       3894                  169 2024-01-20 08:31:00+00:00
4         orders    log.row_count                25       2934                  142 2024-01-20 08:31:00+00:00
5        payment    log.row_count                25       2427                  111 2024-01-20 08:31:00+00:00
6          carts    log.row_count                24       1650                  150 2024-01-20 08:31:00+00:00
7       shipping    log.row_count                25       1467                   71 2024-01-20 08:31:00+00:00
8       carts-db    log.row_count                24         25                    2 2024-01-20 08:32:00+00:00
9     session-db    log.row_count                 5         25                    5 2024-01-20 08:33:00+00:00
10      carts-db  log.error_count                24          0                    0                       NaT
11         carts  log.error_count                24          0                    0                       NaT
12       payment  log.error_count                25          0                    0                       NaT
13        orders  log.error_count                25          0                    0                       NaT
14     catalogue  log.error_count                25          0                    0                       NaT
15     front-end  log.error_count                25          0                    0                       NaT
16    session-db  log.error_count                 5          0                    0                       NaT
17  queue-master  log.error_count                25          0                    0                       NaT
18      shipping  log.error_count                25          0                    0                       NaT
19          user  log.error_count                25          0                    0                       NaT,      cmdb_id log_name  points_in_window  sum_value  max_value_in_window      earliest_nonzero_utc
0       user      ALL                50      14171                  644 2024-01-20 08:31:00+00:00
1  catalogue      ALL                50       3894                  169 2024-01-20 08:31:00+00:00
2     orders      ALL                50       2934                  142 2024-01-20 08:31:00+00:00
3    payment      ALL                50       2427                  111 2024-01-20 08:31:00+00:00
4      carts      ALL                48       1650                  150 2024-01-20 08:31:00+00:00)```
```